In [1]:
import numpy as np
from math import sqrt
from scipy.linalg import expm

In [2]:
np.random.seed(56)
n_qb = 3
n_b = 1<<n_qb

a = np.random.rand(n_b, n_b)
H = (a + a.T) / 2  # Make it symmetric
print("H:\n", H)
print("Is Hermitian:", (H == np.conj(H).T).all())
eigvals, eigvecs = np.linalg.eigh(H)

eigvals = np.random.rand(n_b) * 3
eigvals = np.sort(eigvals)
eigvals = eigvals - eigvals[0]
# eigvals[1:] += 1

H = eigvecs @ np.diag(eigvals) @ eigvecs.T
# print("Modified H:\n", H)

for i in range(len(eigvals)):
    print(f"Eigenvalue {i}: {eigvals[i]}")
    print(f"Eigenvector {i}:\n{eigvecs[:, i]}\n")

ground_state = eigvecs[:, 0]
print("Ground state:\n", ground_state)

# print(np.linalg.eigh(H))
# print(np.linalg.eig(H))

b = np.random.rand(n_b)
b = b / np.linalg.norm(b)
print("Vector b:\n", b)

spectral_gap = eigvals[1] - eigvals[0]
spectral_radius = max(abs(eigvals))
print("\nSpectral gap:", spectral_gap)
print("Spectral radius:", spectral_radius)

ss = spectral_gap / (12 * spectral_radius**3)
F0 = abs(b @ ground_state)**2
print("F0:", F0)
q = 1 - ss * F0 * spectral_gap
print("q:", q)
print("Recommended step size (s):", ss)
assert spectral_radius > 1

H:
 [[0.98419185 0.60535684 0.76172108 0.26209364 0.43185054 0.69955134
  0.13663526 0.30567345]
 [0.60535684 0.39165846 0.29572537 0.32655587 0.59199509 0.12636866
  0.41206312 0.27557321]
 [0.76172108 0.29572537 0.44086832 0.81330121 0.83383871 0.74935803
  0.50156741 0.23917602]
 [0.26209364 0.32655587 0.81330121 0.72956789 0.59718664 0.56441096
  0.41625456 0.51353732]
 [0.43185054 0.59199509 0.83383871 0.59718664 0.4011665  0.60826709
  0.48206403 0.72926474]
 [0.69955134 0.12636866 0.74935803 0.56441096 0.60826709 0.25925376
  0.56219998 0.40411403]
 [0.13663526 0.41206312 0.50156741 0.41625456 0.48206403 0.56219998
  0.46898145 0.16724097]
 [0.30567345 0.27557321 0.23917602 0.51353732 0.72926474 0.40411403
  0.16724097 0.53527332]]
Is Hermitian: True
Eigenvalue 0: 0.0
Eigenvector 0:
[ 0.30540055 -0.3467468  -0.53692428  0.22485379  0.56175294 -0.20960435
  0.13930364 -0.26219153]

Eigenvalue 1: 0.3999622035194259
Eigenvector 1:
[ 0.21399313 -0.28221316  0.38815714 -0.04788439 -0

In [3]:
def commutator(A, B):
    return A @ B - B @ A

## Exact imaginary time evolution

In [4]:
# tau = 200
# tau = 0.2
N = 6000
tau = ss * N
N = 100

print("tau:", tau)


# e^(-tau H)
exp_H = expm(-tau * H)
print("Exponential of -tau * H:\n", exp_H)

result = exp_H @ b
result = result / np.linalg.norm(result)
print("Result of normed exp(-tau * H) @ b:", result)

tau: 8.783378705104626
Exponential of -tau * H:
 [[ 0.09465216 -0.10769465 -0.16153468  0.06839065  0.17065974 -0.06862811
   0.04470152 -0.07841871]
 [-0.10769465  0.12260753  0.18290769 -0.07756151 -0.19366286  0.07878009
  -0.05111577  0.08875764]
 [-0.16153468  0.18290769  0.29284206 -0.12133194 -0.30308697  0.10413368
  -0.07096718  0.14371187]
 [ 0.06839065 -0.07756151 -0.12133194  0.05066457  0.12643975 -0.04608092
   0.03087885 -0.05929505]
 [ 0.17065974 -0.19366286 -0.30308697  0.12643975  0.31621338 -0.11487182
   0.07685785 -0.14835779]
 [-0.06862811  0.07878009  0.10413368 -0.04608092 -0.11487182  0.05961028
  -0.03641902  0.0494208 ]
 [ 0.04470152 -0.05111577 -0.07096718  0.03087885  0.07685785 -0.03641902
   0.02277506 -0.03394066]
 [-0.07841871  0.08875764  0.14371187 -0.05929505 -0.14835779  0.0494208
  -0.03394066  0.07072533]]
Result of normed exp(-tau * H) @ b: [-0.30066097  0.34061051  0.54500799 -0.22572899 -0.56468203  0.19401458
 -0.13198653  0.26771677]


## First order approximation from differential equation of DBF

In [5]:
# densi = np.outer(b, b)
# Q = expm(tau*commutator(densi, H))
# result = Q @ b
s = tau / N
bb = b.copy()
for i in range(N):
    densi = np.outer(bb, bb)
    Q = expm(s * commutator(densi, H))
    bb = Q @ bb
result = bb
print("Result:", result)
print("Norm:", np.linalg.norm(result))

Result: [-0.30071709  0.34066422  0.54496245 -0.22574094 -0.56462379  0.1941461
 -0.13206941  0.26765464]
Norm: 1.0


## DB-QITE (via group commutator)

In [7]:
s = tau / N
bb = b.copy()
for i in range(N):
    densi = np.outer(bb, bb)
    E1 = expm(-1j*sqrt(s)*H)
    E2 = expm(1j*sqrt(s)*densi)
    E3 = expm(1j*sqrt(s)*H)
    bb = (np.exp(-1j*sqrt(s)) * E3) @ E2 @ E1 @ bb
    print(f"iter {i+1}")
    F_k = abs(bb @ ground_state)**2
    print(f"F_{i+1}:", F_k)
    print("bb:", bb)
    lower_bound = 1 - q**(i)
    print(f"Lower bound for F_{i+1}:", lower_bound)
    print()
print("Result of DB-QITE:", bb)
print("Abs of DB-QITE result:", np.abs(bb) * np.sign(bb.real))
print("Norm of DB-QITE result:", np.linalg.norm(bb))


iter 1
F_1: 0.0606697220986777
bb: [0.345297  +5.60081210e-06j 0.32587881-5.27724388e-03j
 0.27960035-1.20192365e-02j 0.01459945+3.44147934e-03j
 0.13231341+1.94231066e-02j 0.28749495+5.10238949e-04j
 0.43976241-7.37466640e-03j 0.63386341-1.52784672e-02j]
Lower bound for F_1: 0.0

iter 2
F_2: 0.0854566608223503
bb: [ 3.35103328e-01+0.00191152j  3.22912362e-01-0.01147195j
  2.88732810e-01-0.02803593j -1.37517177e-04+0.00820369j
  7.86463111e-02+0.04644436j  2.76600115e-01+0.0024901j
  4.49452812e-01-0.01491153j  6.54084152e-01-0.03310015j]
Lower bound for F_2: 2.5304250909630177e-05

iter 3
F_3: 0.12166422846987791
bb: [ 0.32544347+0.00804387j  0.3214125 -0.01737514j  0.30131437-0.04807034j
 -0.01536627+0.01467487j  0.01892829+0.08372565j  0.26542548+0.0077382j
  0.46184981-0.02028326j  0.67915891-0.05093351j]
Lower bound for F_3: 5.060786151411811e-05

iter 4
F_4: 0.17637343731278557
bb: [ 0.31568935+0.02147669j  0.32140179-0.02145006j  0.31861075-0.07207399j
 -0.03177009+0.02302399j -